In [26]:
import pandas as pd
df = pd.read_csv('retailcleaned.csv',header=None)

C:\Users\SHOAIB\AppData\Local\Temp\ipykernel_26632\3026557978.py:2: DtypeWarning: Columns (0: 0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('retailcleaned.csv',header=None)


In [28]:
# Naming the columns of the DataFrame
df.columns = ['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country','SourceSheet']

In [29]:
df['InvoiceDate']= pd.to_datetime(df['InvoiceDate'])
print(df.shape)
df.info()

(1033036, 9)
<class 'pandas.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   InvoiceNo    1033036 non-null  object        
 1   StockCode    1033036 non-null  str           
 2   Description  1028761 non-null  str           
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   UnitPrice    1033036 non-null  float64       
 6   CustomerID   797885 non-null   float64       
 7   Country      1033036 non-null  str           
 8   SourceSheet  1033036 non-null  str           
dtypes: datetime64[ns](1), float64(2), int64(1), object(1), str(4)
memory usage: 70.9+ MB


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1033036 entries, 0 to 1033035
Data columns (total 9 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   InvoiceNo    1033036 non-null  object        
 1   StockCode    1033036 non-null  str           
 2   Description  1028761 non-null  str           
 3   Quantity     1033036 non-null  int64         
 4   InvoiceDate  1033036 non-null  datetime64[ns]
 5   UnitPrice    1033036 non-null  float64       
 6   CustomerID   797885 non-null   str           
 7   Country      1033036 non-null  str           
 8   SourceSheet  1033036 non-null  str           
dtypes: datetime64[ns](1), float64(1), int64(1), object(1), str(5)
memory usage: 70.9+ MB


In [36]:
# FILTER AND CALCULATE RECENCY, FREQUENCY, MONETARY
rfm_data = df[(df['CustomerID'].notna()) & (df['Quantity'] > 0)].copy()
snapshot_date = pd.to_datetime('2011-12-10')

rfm = rfm_data.groupby('CustomerID').agg(
    Recency=('InvoiceDate', lambda x: (snapshot_date - x.max()).days),
    Frequency=('InvoiceNo', 'nunique'),
    Monetary=('Quantity', lambda x: (x * rfm_data.loc[x.index, 'UnitPrice']).sum())
).reset_index()

In [38]:
# SCORE CUSTOMERS AND ASSIGN BUSINESS SEGMENTS
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5]).astype(int)

def segment_customer(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'Champions'
    elif row['R_Score'] >= 3 and row['F_Score'] >= 3:
        return 'Loyal'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 3:
        return 'At Risk'
    elif row['R_Score'] >= 4 and row['F_Score'] <= 2:
        return 'New'
    elif row['R_Score'] == 3 and row['F_Score'] <= 2:
        return 'Needs Attention'
    else:
        return 'Lost'

rfm['Segment'] = rfm.apply(segment_customer, axis=1)
rfm['IsChurned'] = rfm['Recency'] > 90

In [41]:
# MERGE AND EXPORT TO EXCEL
df_dashboard = df[(df['CustomerID'].notna()) & (df['Quantity'] > 0)].copy()
df_dashboard['InvoiceMonth'] = df_dashboard['InvoiceDate'].dt.to_period('M').astype(str)
df_dashboard['Revenue'] = df_dashboard['Quantity'] * df_dashboard['UnitPrice']

df_dashboard = df_dashboard.merge(
    rfm[['CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'Segment', 'IsChurned']],
    on='CustomerID',
    how='left'
)

df_dashboard.to_excel('ecommerce_dashboard_data.xlsx', index=False)